<a href="https://colab.research.google.com/github/Moksh008/Assignment-Parameter-passing/blob/main/pdf_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Please run this cell, to install the dependencies
!pip install -q pypdf openai pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 9.3 MB/s eta 0:00:00



# Speak with PDF Assignment

**Assignment type:** RAG mini-app with three required functions  
**File to submit:** `pdf_ai.ipynb` (the notebook produced from this spec)  
**Language:** Python 3.x (Codio / Jupyter)  
**Time estimate:** 2–3 hours

---

## Objective

Build a **“Speak with PDF” RAG pipeline** that runs inside a Jupyter notebook and is compatible with **NBGrader**. You will implement **three functions**:

1) `parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120) -> List[str]`  
2) `embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict`  
3) `answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini") -> str`

**Grading will use OpenAI keys.** Learners must provide valid `OPENAI_API_KEY` (and Pinecone credentials if using Pinecone).

---

## What you will build

A tiny RAG system that:
- loads a PDF,
- extracts and **chunks** text,
- builds a **vector index** from OpenAI embeddings (Pinecone recommended; in‑memory/FAISS acceptable),
- answers a natural-language query by retrieving top‑k relevant chunks and generating an answer with an OpenAI chat model.

This mirrors production flow (PDF → chunks → embeddings → index → retrieval → answer).


---

## Tools & Libraries

- `openai` (for embeddings + chat completions)
- `pypdf` **or** `pdfplumber` (choose one) for text extraction
- Vector store: **Pinecone** (recommended) via `pinecone-client`; you may alternatively use **FAISS** or a simple in‑memory index for development
- `numpy`, `re`, `math`, `hashlib` (as needed)

> You may import additional libraries if helpful. Use your judgment.

**Environment variables expected (for grading):**
- `OPENAI_API_KEY`
- If using Pinecone: `PINECONE_API_KEY`, `PINECONE_ENV` (or project config), and an index name you create in notebook (e.g., `pdf-mini-app`).



---

## Function Requirements
You'll have to implment the following functionality down below

### 1) `parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120) -> list[str]`

**Inputs**
- `pdf_path: str` — path to a readable PDF file (e.g., `data/sample.pdf`)
- `chunk_size: int` (default 800) — approximate number of characters per chunk
- `overlap: int` (default 120) — number of overlapping characters between adjacent chunks

**Output**
- `List[str]` — ordered, non-empty chunks of plain text. No chunk should exceed `chunk_size + 20` characters (allowing minor boundary variance).

**Behavioral constraints**
- Strip excessive whitespace and normalize newlines.
- Deterministic chunk boundaries given the same inputs.
- Apply overlap between chunks (except the first).

**Recommended approach**
1. Extract all PDF text into a single string using `pypdf` **or** `pdfplumber`.
2. Normalize whitespace.
3. Greedy chunk by character count with the requested overlap.



---

### 2) `embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict`

**Purpose**  
Create embeddings with **OpenAI** and store them in a vector index (Pinecone preferred).

**Inputs**
- `chunks: List[str]` — the output of `parse_pdf_to_chunks`
- `use_pinecone: bool` — if `True`, upsert to Pinecone; if `False`, store locally (FAISS or simple in‑memory array search)
- `model: str` — OpenAI embedding model name (e.g., `"text-embedding-3-small"`)

**Output**
- `dict` with fields (minimum):
  - `"chunks"`: `List[str]`
  - `"index_backend"`: `"pinecone"` or `"local"`
  - `"index_ref"`: backend-specific handle (e.g., Pinecone index name / namespace, or local matrix + ids)
  - `"embed_model"`: embedding model used
  - `"dim"`: embedding dimensionality

**Notes**
- When `use_pinecone=True`: create (if needed) and upsert vectors with stable IDs (e.g., `doc-0`, `doc-1`, …). Keep a mapping from ID → original chunk text in the dict.
- When `use_pinecone=False`: store embeddings in memory (e.g., NumPy matrix) along with their chunk IDs for cosine‑similarity search.



---

### 3) `answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini") -> str`

**Purpose**  
Retrieve relevant chunks and generate an answer using an OpenAI chat model.

**Inputs**
- `question: str`
- `index_obj: dict` — the output from `embed_and_index`
- `top_k: int` — number of chunks to retrieve
- `gpt_model: str` — OpenAI chat model (e.g., `"gpt-4o-mini"`)

**Output**
- `str` — the final answer.

**Required behavior**
- Embed the **question** using the **same embedding model** used to build the index.
- Retrieve **top_k** chunks by cosine similarity (Pinecone query or local similarity search).
- Construct a concise prompt:
  - System: “You are a helpful assistant. Use ONLY the provided context to answer. If not answerable from context, respond: ‘I don’t know based on the document.’”
  - User: include the `question` and a `Context:` section concatenating the retrieved chunks (with simple separators).
- Call the OpenAI chat completion endpoint and **return the assistant’s message string**.



In [2]:
# This cell may be run by the grader
import os, re, math, hashlib
import numpy as np

In [3]:
import os
from getpass import getpass

# SECURITY: the key is never hard-coded. It is read from the environment if already set,
# otherwise you are prompted for it (getpass hides what you type and nothing is saved in the notebook).
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# Pinecone is optional. Leave it unset to use the in-memory NumPy index.
os.environ.setdefault("PINECONE_API_KEY", "")

Enter your OpenRouter API key: ··········


''

In [4]:
OPENAI_API_KEY = OPENROUTER_API_KEY
assert OPENAI_API_KEY, "Please set OPENAI_API_KEY"
USE_PINECONE = bool(os.getenv("PINECONE_API_KEY"))

from openai import OpenAI

# OpenRouter exposes an OpenAI-compatible API, so the regular `openai` client works:
# we only point it at OpenRouter's base URL.
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENAI_API_KEY)

# OpenRouter model IDs are "<provider>/<model>". Bare OpenAI names from the spec are mapped below.
def _or_model(name: str) -> str:
    return name if "/" in name else f"openai/{name}"

print("Client ready. Using Pinecone:", USE_PINECONE)

Client ready. Using Pinecone: False


<center><h1>--------------- Student Code ---------------</h1></center>

### B. Implement `parse_pdf_to_chunks`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [5]:
from typing import List
from pypdf import PdfReader


def _normalize_text(text: str) -> str:
    """Clean raw PDF text: unify newlines, re-join hyphenated line breaks, collapse whitespace."""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00a0", " ").replace("\x00", "")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)   # "embed-\nding" -> "embedding"
    text = re.sub(r"\s+", " ", text)                # newlines, tabs, repeated spaces -> one space
    return text.strip()


def parse_pdf_to_chunks(pdf_path: str, chunk_size: int = 800, overlap: int = 120) -> List[str]:
    """
    Extract text from a PDF and split it into ordered, overlapping character chunks.

    - Every chunk is at most `chunk_size` characters (well inside the chunk_size + 20 limit).
    - Adjacent chunks share `overlap` characters, so a sentence cut at a boundary still
      appears whole in at least one chunk.
    - Boundaries are deterministic: they depend only on the text and the two parameters.
    - Where possible a chunk ends at a space (never cuts a word), searching only the last
      20% of the window so chunks stay close to the requested size.
    """
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(pdf_path)

    # 1. Extract text page by page. Empty/scanned pages return None or "" and are skipped.
    reader = PdfReader(pdf_path)
    pages = []
    for page in reader.pages:
        try:
            page_text = page.extract_text() or ""
        except Exception:
            page_text = ""   # a single unreadable page should not break the whole document
        if page_text.strip():
            pages.append(page_text)

    # 2. Normalize whitespace into a single string.
    text = _normalize_text("\n".join(pages))
    if not text:
        return []

    # 3. Greedy chunking by character count with overlap.
    chunks, start, n = [], 0, len(text)
    min_end_offset = int(chunk_size * 0.8)
    while start < n:
        end = min(start + chunk_size, n)
        if end < n:
            space = text.rfind(" ", start + min_end_offset, end)
            if space != -1:
                end = space                          # end on a word boundary
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= n:
            break
        start = max(end - overlap, start + 1)        # step back by `overlap`, always move forward
    return chunks

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `parse_pdf_to_chunks`.
 -  If the function is not found, you answer will be rejected by the system

### C. Implement `embed_and_index`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [6]:
def _embed_texts(texts: List[str], model: str, batch_size: int = 64) -> np.ndarray:
    """Embed a list of strings through OpenRouter's embeddings endpoint. Returns an (n, dim) float32 array."""
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(model=_or_model(model), input=batch)
        # Sort by the returned index so vectors line up with the inputs.
        vectors.extend(d.embedding for d in sorted(resp.data, key=lambda d: d.index))
    return np.asarray(vectors, dtype=np.float32)


def _l2_normalize(mat: np.ndarray) -> np.ndarray:
    """Scale rows to unit length so a dot product equals cosine similarity."""
    norms = np.linalg.norm(mat, axis=-1, keepdims=True)
    return mat / np.clip(norms, 1e-12, None)


def embed_and_index(chunks: List[str], use_pinecone: bool = True,
                    model: str = "text-embedding-3-small") -> dict:
    """
    Embed every chunk and store the vectors in Pinecone or a local NumPy index.

    Returns a dict with: chunks, ids, id_to_chunk, index_backend, index_ref, embed_model, dim.
    IDs are stable: chunk i always gets id "doc-i".
    """
    if not chunks:
        raise ValueError("No chunks to index - was the PDF parsed correctly?")

    ids = [f"doc-{i}" for i in range(len(chunks))]
    embeddings = _embed_texts(chunks, model)
    dim = int(embeddings.shape[1])

    result = {
        "chunks": list(chunks),
        "ids": ids,
        "id_to_chunk": dict(zip(ids, chunks)),
        "embed_model": model,
        "dim": dim,
    }

    # Pinecone is used only when asked for AND a key is available; otherwise fall back to local.
    if use_pinecone and os.getenv("PINECONE_API_KEY"):
        from pinecone import Pinecone, ServerlessSpec
        pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
        index_name = "pdf-mini-app"
        if index_name not in pc.list_indexes().names():
            pc.create_index(name=index_name, dimension=dim, metric="cosine",
                            spec=ServerlessSpec(cloud="aws", region="us-east-1"))
        index = pc.Index(index_name)
        namespace = hashlib.md5("".join(chunks).encode()).hexdigest()[:12]  # one namespace per document
        vectors = [{"id": ids[i], "values": embeddings[i].tolist(),
                    "metadata": {"chunk": chunks[i], "order": i}} for i in range(len(chunks))]
        for i in range(0, len(vectors), 100):
            index.upsert(vectors=vectors[i:i + 100], namespace=namespace)
        result["index_backend"] = "pinecone"
        result["index_ref"] = {"index_name": index_name, "namespace": namespace, "index": index}
    else:
        # Local backend: keep a unit-normalized matrix so retrieval is a single matrix-vector product.
        result["index_backend"] = "local"
        result["index_ref"] = {"matrix": _l2_normalize(embeddings), "ids": ids}
    return result

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `embed_and_index`.
 -  If the function is not found, you answer will be rejected by the system

### D. Implement `answer_query`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [7]:
SYSTEM_PROMPT = ("You are a helpful assistant. Use ONLY the provided context to answer. "
                 "If not answerable from context, respond: 'I don't know based on the document.'")


def retrieve(question: str, index_obj: dict, top_k: int = 3):
    """
    Return [(id, score, chunk_text), ...] for the top_k chunks by cosine similarity.
    The question is embedded with the SAME model used for the chunks.
    Ties are broken deterministically by chunk order (lower id first).
    """
    q_vec = _embed_texts([question], index_obj["embed_model"])[0]
    top_k = max(1, min(top_k, len(index_obj["chunks"])))

    if index_obj["index_backend"] == "pinecone":
        ref = index_obj["index_ref"]
        res = ref["index"].query(vector=q_vec.tolist(), top_k=top_k,
                                 namespace=ref["namespace"], include_metadata=True)
        hits = [(m.id, float(m.score), m.metadata["chunk"]) for m in res.matches]
        return sorted(hits, key=lambda h: (-h[1], int(h[0].split("-")[1])))

    ref = index_obj["index_ref"]
    scores = ref["matrix"] @ _l2_normalize(q_vec)            # cosine similarity with every chunk
    order = np.argsort(-scores, kind="stable")[:top_k]       # stable sort = tie-break by position
    return [(ref["ids"][i], float(scores[i]), index_obj["chunks"][i]) for i in order]


def answer_query(question: str, index_obj: dict, top_k: int = 3,
                 gpt_model: str = "gpt-4o-mini") -> str:
    """Retrieve the top_k chunks and ask the chat model to answer using only that context."""
    hits = retrieve(question, index_obj, top_k)
    context = "\n\n---\n\n".join(f"[{cid}] {text}" for cid, _, text in hits)
    user_msg = f"Question: {question}\n\nContext:\n{context}"
    try:
        resp = client.chat.completions.create(
            model=_or_model(gpt_model),
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": user_msg}],
            temperature=0,
        )
        return (resp.choices[0].message.content or "").strip()
    except Exception as e:
        return f"[Error calling chat model: {e}]"


def vanilla_answer(question: str, gpt_model: str = "gpt-4o-mini") -> str:
    """Baseline for the comparison: same model, same question, NO retrieved context."""
    resp = client.chat.completions.create(
        model=_or_model(gpt_model),
        messages=[{"role": "system", "content": "You are a helpful assistant."},
                  {"role": "user", "content": question}],
        temperature=0,
    )
    return (resp.choices[0].message.content or "").strip()

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `answer_query`.
 -  If the function is not found, you answer will be rejected by the system

### E. Demo, chunk-size experiment and Vanilla vs RAG comparison

**Sample PDF:** *Attention Is All You Need* (Vaswani et al., 2017, arXiv:1706.03762), the paper that introduced the Transformer.
It is downloaded into `data/sample.pdf` below.

In [8]:
# Download the sample PDF and run the three functions end to end.
import urllib.request, time
os.makedirs("data", exist_ok=True)
PDF_PATH = "data/sample.pdf"
if not os.path.exists(PDF_PATH):
    urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", PDF_PATH)

chunks = parse_pdf_to_chunks(PDF_PATH, chunk_size=800, overlap=120)
print(f"Parsed {len(chunks)} chunks | longest = {max(map(len, chunks))} chars")
print("Deterministic:", chunks == parse_pdf_to_chunks(PDF_PATH, 800, 120))
print("\nChunk 0 preview:\n", chunks[0][:300], "...")

t0 = time.time()
index_obj = embed_and_index(chunks, use_pinecone=USE_PINECONE)
print(f"\nIndexed {len(index_obj['chunks'])} chunks | backend = {index_obj['index_backend']} "
      f"| dim = {index_obj['dim']} | {time.time() - t0:.1f}s")

print("\nExample answer:", answer_query("What is the Transformer?", index_obj))

Parsed 59 chunks | longest = 799 chars
Deterministic: True

Chunk 0 preview:
 Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Par ...

Indexed 59 chunks | backend = local | dim = 1536 | 2.2s

Example answer: The Transformer is a model architecture that relies entirely on self-attention mechanisms to compute representations of its input and output, without using sequence-aligned recurrent neural networks (RNNs) or convolution. It features an encoder-decoder structure, where the encoder maps an input sequence to continuous representations, and the decoder generates an output sequence one element at a time. The Transformer allows for significant parallelization and has achieved state-of-the-art results in tasks such as translation.


In [9]:
# Chunk-size experiment: how chunk size changes the number of chunks and what gets retrieved.
probe = "What BLEU score did the big Transformer get on English-to-German translation?"
experiment_indexes = {}
for size, ov in [(300, 50), (800, 120), (1500, 200)]:
    ch = parse_pdf_to_chunks(PDF_PATH, chunk_size=size, overlap=ov)
    t0 = time.time()
    idx = embed_and_index(ch, use_pinecone=False)
    experiment_indexes[size] = idx
    top = retrieve(probe, idx, top_k=1)[0]
    print(f"chunk_size={size:<5} overlap={ov:<4} -> {len(ch):>3} chunks, indexed in {time.time()-t0:.1f}s, "
          f"top score {top[1]:.3f} ({top[0]})")
    print("   top chunk:", top[2][:200].replace("\n", " "), "...")
    print("   answer   :", answer_query(probe, idx, top_k=3), "\n")

chunk_size=300   overlap=50   -> 161 chunks, indexed in 2.4s, top score 0.738 (doc-92)
   top chunk: sh-to-German translation task, the big transformer model (Transformer (big) in Table 2) outperforms the best previously reported models (including ensembles) by more than 2.0 BLEU, establishing a new  ...
   answer   : The big Transformer achieved a BLEU score of 28.4 on English-to-German translation. 

chunk_size=800   overlap=120  ->  59 chunks, indexed in 1.0s, top score 0.640 (doc-33)
   top chunk: we use a rate of Pdrop = 0.1. Label Smoothing During training, we employed label smoothing of value ϵls = 0.1 [36]. This hurts perplexity, as the model learns to be more unsure, but improves accuracy  ...
   answer   : The big Transformer achieved a BLEU score of 28.4 on the WMT 2014 English-to-German translation task. 

chunk_size=1500  overlap=200  ->  31 chunks, indexed in 0.9s, top score 0.651 (doc-16)
   top chunk: m line of table 3), step time was 1.0 seconds. The big models were tr

In [10]:
# Comparative playtest: 3 domain-specific questions, Vanilla LLM vs RAG-augmented LLM.
QUESTIONS = [
    "In the paper 'Attention Is All You Need', what BLEU scores did the big Transformer achieve on "
    "WMT 2014 English-to-German and English-to-French, and how long did it train and on what hardware?",
    "In 'Attention Is All You Need', what values of d_model, d_ff, number of heads h and d_k are used "
    "in the base model, and why is the dot product scaled by 1/sqrt(d_k)?",
    "In 'Attention Is All You Need', what learning-rate schedule, warmup steps, and regularisation "
    "(dropout rate and label smoothing value) were used for training?",
]

results = []
for i, q in enumerate(QUESTIONS, 1):
    t0 = time.time(); v = vanilla_answer(q); tv = time.time() - t0
    t0 = time.time(); hits = retrieve(q, index_obj, top_k=3); r = answer_query(q, index_obj, top_k=3); tr = time.time() - t0
    results.append({"question": q, "vanilla": v, "rag": r, "sources": [h[0] for h in hits],
                    "t_vanilla": tv, "t_rag": tr})
    print("=" * 100)
    print(f"Q{i}: {q}\n")
    print(f"--- VANILLA LLM ({tv:.1f}s) ---\n{v}\n")
    print(f"--- RAG-AUGMENTED LLM ({tr:.1f}s) | retrieved: "
          + ", ".join(f"{cid} ({s:.3f})" for cid, s, _ in hits) + f" ---\n{r}\n")

Q1: In the paper 'Attention Is All You Need', what BLEU scores did the big Transformer achieve on WMT 2014 English-to-German and English-to-French, and how long did it train and on what hardware?

--- VANILLA LLM (1.8s) ---
In the paper "Attention Is All You Need," the authors reported BLEU scores for the big Transformer model on the WMT 2014 English-to-German and English-to-French translation tasks. The big Transformer achieved a BLEU score of 28.4 on the English-to-German task and a BLEU score of 41.8 on the English-to-French task.

Regarding training, the model was trained for about 3 to 4 days on 8 NVIDIA P100 GPUs. This setup allowed for efficient training of the model using the proposed architecture.

--- RAG-AUGMENTED LLM (2.7s) | retrieved: doc-34 (0.722), doc-33 (0.705), doc-2 (0.690) ---
The big Transformer achieved a BLEU score of 28.4 on the WMT 2014 English-to-German translation task and a BLEU score of 41.0 on the English-to-French translation task. It trained for 3.5 day

In [11]:
# Out-of-scope check: RAG should refuse instead of inventing an answer.
oos = "What stock price did Google reach in 2017 according to this paper?"
print("VANILLA:", vanilla_answer(oos)[:400], "\n")
print("RAG    :", answer_query(oos, index_obj))

VANILLA: I'm sorry, but I cannot access external documents or papers directly. However, I can provide general information about Google's stock price in 2017. If you have specific details or excerpts from the paper, feel free to share them, and I can help you interpret or analyze that information! 

RAG    : I don't know based on the document.


In [12]:
# Save the comparison so it can be attached to the submission (no API key is written anywhere).
import json
with open("rag_vs_vanilla_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved rag_vs_vanilla_results.json")

Saved rag_vs_vanilla_results.json


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  If the function is not found, you answer will be rejected by the system

---

## Hidden Tests (run by grader)

Hidden tests will verify that:

- `parse_pdf_to_chunks` applies overlap correctly, is deterministic, and honors size bounds.
- `embed_and_index` calls OpenAI embeddings and builds a valid index (Pinecone or local), with stable IDs and correct dimensionality/shape.
- Retrieval uses cosine similarity and returns the expected top‑k order on crafted inputs.
- `answer_query` embeds the question with the same model used for chunks, constructs a context‑bounded prompt, calls OpenAI chat completions, and returns a clean string.
- No hard‑coded secrets; keys must be read from environment.
- Notebook runs top‑to‑bottom reproducibly when keys are present.



---

## Rubric (100 pts)

- **PDF parsing & chunking (30 pts)**
  - Correct extraction and normalization (10)
  - Deterministic overlapping chunks within size bounds (15)
  - Edge cases (empty pages, weird whitespace) (5)

- **Embeddings & indexing (35 pts)**
  - Correct OpenAI embedding usage and shapes (15)
  - Proper index structure + metadata (10)
  - Stable IDs and retrievability (10)

- **Query flow & answer (25 pts)**
  - Correct cosine similarity retrieval and stable tie‑break (10)
  - Prompt enforces context‑only answering; sensible output (15)

- **Code quality & constraints (10 pts)**
  - Clean, readable, documented code (5)
  - Secure key handling; graceful error handling (5)



---

## (Optional) Pinecone/OpenAI Notes

- For Pinecone: create an index (e.g., 384 or 1536 dim depending on embedding model) and upsert vectors with metadata (`{"chunk": "...", "order": i}`).
- For local dev without Pinecone: store embeddings in a NumPy matrix and do cosine similarity. Grading is OK with either backend as long as OpenAI embeddings + chat are used.
- Keep your mapping from vector IDs to original chunk text; you’ll need it for context construction.



---

## Submission

Submit your completed notebook. Ensure it runs top‑to‑bottom without modification in a clean environment **with valid API keys set**. The grader will run hidden tests using OpenAI for embeddings and chat.


---

<h2><center> Completing your test </center></h2>

- Once you have completed your test and answered all the questions:
  
- You can open the guide using the button located on the far right of your screen.

- The button will look like this image:  
  ![](additional_files/End_guide_button.png)

- Clicking on this button will allow you to mark the test as complete using the button shown below:  
  ![](additional_files/completion_button.png)

- Please click on **Mark as Completed** to end and submit your test.

- Thank you!
